# Health Recommendation System — Rule Restoration Engine

This notebook implements a **rule restoration system** that reverse-engineers the hidden logic of a health recommendation engine from its output text alone — without access to the original rule source code.

**The core problem:** Given a dataset where health tips are generated by an unknown rule engine based on 8 health dimension values, can we fully reconstruct which conditions trigger which tips?

**Approach:**
1. `decompose_rules()` — For each `(tip, dimension)` pair, compute tip appearance frequency across all value range combinations. High variance → dimension controls this tip.
2. `classify_abcd()` — Assign each tip a global type based on its cross-dimension trigger pattern:
   - **Type A** — Triggered exclusively by one dimension (background freq ≈ 0)
   - **Type B** — Modulated by one dimension but present at baseline (background freq > 0)
   - **Type C** — Jointly controlled by two or more dimensions (coupled logic)
   - **Type D** — Constant tip, appears regardless of all dimension values

---
> ⚠️ **Data Notice:** The original dataset is proprietary and cannot be shared publicly.  
> A mock sample (`data_sample.xlsx`) with identical column structure is provided for reproduction.  
> **Outputs below were produced on the full dataset (746,496 rows) and are preserved for reference.**


In [1]:
import pandas as pd
import numpy as np
import pickle
from collections import defaultdict

with open("final_base_tips.pkl", "rb") as f:
    base_tips = list(pickle.load(f))

def decompose_rules(file_path, base_tips):
    """
    For each dimension and each base tip, compute:
    - active ranges (high-response ranges)
    - active frequency
    - background frequency
    using variance + mean-based logic.

    A tip is considered controlled by a dimension if its appearance
    frequency varies significantly (variance > 10) across that
    dimension's value range combinations.
    """
    dimensions = ['nutrition', 'obesity', 'sleep', 'depression',
                  'wellness', 'anti_stress', 'anti_smoke', 'movement']

    # { dim: { tip: { 'active_conditions': [], 'active_freq': x, 'background_freq': y, 'variance': v } } }
    dim_tip_profiles = defaultdict(dict)

    print("--- Starting Automated Logic Decomposition ---")

    for dim in dimensions:
        cols = [f'dif_{dim}', f'c_val_{dim}', 'recommendations']
        df = pd.read_excel(file_path, usecols=cols)
        df['range'] = df[f'dif_{dim}'].astype(str) + " | " + df[f'c_val_{dim}'].astype(str)

        range_counts = df['range'].value_counts()
        unique_ranges = range_counts.index.tolist()

        for tip in base_tips:
            freqs = []
            range_to_freq = {}
            for r in unique_ranges:
                subset = df[df['range'] == r]
                f = (
                    subset['recommendations']
                    .astype(str)
                    .str.contains(tip, regex=False, na=False)
                    .sum() / len(subset)
                ) * 100
                freqs.append(f)
                range_to_freq[r] = f

            variance = np.var(freqs)

            # Threshold chosen empirically: constant tips have variance ≈ 0,
            # rule-driven tips show variance ranging from 50 to 2500
            if variance > 10:
                avg_f = np.mean(freqs)
                active_ranges   = [r for r, f in range_to_freq.items() if f > avg_f + 5]
                inactive_ranges = [r for r, f in range_to_freq.items() if f < avg_f]

                active_freq = np.mean([range_to_freq[r] for r in active_ranges])   if active_ranges   else 0
                bg_freq     = np.mean([range_to_freq[r] for r in inactive_ranges]) if inactive_ranges else 0

                dim_tip_profiles[dim][tip] = {
                    'active_conditions': active_ranges,
                    'active_freq':       round(active_freq, 2),
                    'background_freq':   round(bg_freq, 2),
                    'variance':          round(variance, 2)
                }

        print(f"Dimension [{dim}] decomposition completed. Associated Tips: {len(dim_tip_profiles[dim])}")
        del df  # free memory

    return dim_tip_profiles


---
## ABCD Classification

Once per-dimension profiles are built, each tip is assigned a global type based on its cross-dimension trigger pattern.

| Type | Meaning |
|------|---------|
| **A** | Triggered exclusively by one dimension — background freq ≈ 0, appears only when that dimension is active |
| **B** | Modulated by one dimension but present at baseline — background freq > 0, frequency increases when active |
| **C** | Jointly controlled by two or more dimensions — neither alone fully determines the tip |
| **D** | Constant tip — appears at ~50% rate regardless of all dimension values (global background tip) |


In [2]:
def classify_abcd(file_path, dim_tip_profiles, base_tips):
    """
    Build a global ABCD classification based on per-dimension profiles.
    Output: one row per tip.
    """
    # 1. Per-dimension A/B classification
    #    tip_dim_info[tip][dim] = {'type': 'A'/'B', 'active_freq': x, 'background_freq': y}
    tip_dim_info = defaultdict(dict)

    for dim, tips in dim_tip_profiles.items():
        for tip, info in tips.items():
            act = info['active_freq']
            bg = info['background_freq']

            # A / B rule (based on your verbal definition)
            if bg <= 5 and act >= 30:
                t = 'A'
            elif act > bg > 0:
                t = 'B'
            else:
                continue  # this dim does not clearly define A or B for this tip

            tip_dim_info[tip][dim] = {
                'type': t,
                'active_freq': act,
                'background_freq': bg
            }

    # 2. Read global recommendations once for D-class background frequency
    df_all = pd.read_excel(file_path, usecols=['recommendations'])
    df_all['rec'] = df_all['recommendations'].astype(str)
    total_rows = len(df_all)

    rows = []

    for tip in base_tips:
        dim_info = tip_dim_info.get(tip, {})

        dims_A = [d for d, v in dim_info.items() if v['type'] == 'A']
        dims_B = [d for d, v in dim_info.items() if v['type'] == 'B']

        # --- D class: no dimension triggers at all ---
        if not dims_A and not dims_B:
            hits = df_all['rec'].str.contains(tip, regex=False, na=False).sum()
            bg_global = (hits / total_rows) * 100
            rows.append({
                'tip': tip,
                'type': 'D',
                'triggered_dims': [],
                'active_freq': None,
                'background_freq': round(bg_global, 2)
            })
            continue

        # --- C class: B-type behavior in at least two dimensions ---
        if len(dims_B) >= 2:
            rows.append({
                'tip': tip,
                'type': 'C',
                'triggered_dims': dims_B,   # these are the coupled dimensions
                'active_freq': None,
                'background_freq': None
            })
            continue

        # --- B class: exactly one B dimension, and no special coupling ---
        if len(dims_B) == 1 and len(dims_A) == 0:
            dim = dims_B[0]
            info = dim_info[dim]
            rows.append({
                'tip': tip,
                'type': 'B',
                'triggered_dims': [dim],
                'active_freq': info['active_freq'],
                'background_freq': info['background_freq']
            })
            continue

        # --- A class: at least one A, and no multi-B coupling ---
        if len(dims_A) >= 1 and len(dims_B) == 0:
            # In your current data, each A tip belongs to exactly one dimension.
            main_dim = dims_A[0]
            info = dim_info[main_dim]
            rows.append({
                'tip': tip,
                'type': 'A',
                'triggered_dims': [main_dim],
                'active_freq': info['active_freq'],
                'background_freq': info['background_freq']
            })
            continue

        # --- Mixed rare case: one B + some A (if ever happens, treat as C for safety) ---
        if len(dims_B) == 1 and len(dims_A) >= 1:
            rows.append({
                'tip': tip,
                'type': 'C',
                'triggered_dims': dims_A + dims_B,
                'active_freq': None,
                'background_freq': None
            })
            continue

    return pd.DataFrame(rows)

In [3]:
dim_profiles = decompose_rules('data_sample.xlsx', base_tips)

summary_df = classify_abcd('data_sample.xlsx', dim_profiles, base_tips)

summary_df

--- Starting Automated Logic Decomposition ---
Dimension [nutrition] decomposition completed. Associated Tips: 0
Dimension [obesity] decomposition completed. Associated Tips: 0
Dimension [sleep] decomposition completed. Associated Tips: 3
Dimension [depression] decomposition completed. Associated Tips: 5
Dimension [wellness] decomposition completed. Associated Tips: 4
Dimension [anti_stress] decomposition completed. Associated Tips: 5
Dimension [movement] decomposition completed. Associated Tips: 4


,tip,type,triggered_dims,active_freq,background_freq
0,Chew gum or eat mints when you feel the urge t...,A,[anti_smoke],33.35,0.00
1,Increasing levels of physical activity: engagi...,D,[],NaN,50.05
2,Try to pick up some hobbies that make you happy,A,[depression],40.10,0.00
3,Learn to say no,A,[anti_stress],39.78,0.00
4,Get a journal and write your thoughts out,A,[depression],40.05,0.00
5,Share your feelings with a trusted friend in p...,A,[depression],39.91,0.00
6,Restrict the intake of sugars-sweetened soft d...,D,[],NaN,49.98
7,Avoid who stress you out: If someone is consta...,A,[anti_stress],39.94,0.00
8,Refrain from using electronics or looking at b...,A,[sleep],66.73,0.00
9,Realize that your situation is temporary and t...,A,[depression],39.94,0.00
